<h1> ISO/IEC 25012 </h1>

In [ ]:
import pandas as pd

from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print (device)

<h1> Datasets </h1>

In [39]:
df_phi = pd.read_json (r"C:\Users\Admin\Desktop\ip\How To RAG\dataset-Phi4.json")
df_mistral = pd.read_json (r"C:\Users\Admin\Desktop\ip\How To RAG\dataset-Mistral.json")

<h1> Qualidade Estrutural </h1>

In [40]:
## Colunas
#print (f"{df_phi.columns}\n{df_mistral.columns}")

## Comprimento
#print (f"Phi -> {len(df_phi)}\nMistral -> {len(df_mistral)}")

## Stats
#print (f"{df_phi.describe()}\n{df_mistral.describe()}")
#print (f"{df_phi.info()}\n{df_mistral.info()}")

## Data Cleaning
#print (f"{df_phi.isna().sum()}\n{df_mistral.isna().sum()}") # Valores vazios

## Dups # Maneira crazy de confirmar dups
#dups = [x for x in df_phi["query"]]
#print (len(dups), len(set(dups)))

#### ------------------------------------------------------------------------- ####

#dups = [x for x in df_mistral["query"]]
#print (len(dups), len(set(dups)))

##

<hr>

<h1> Diversidade </h1>


<h3> Distinct-1 </h3>

<p> Mede diretamente repetições no dataset </p>
<p> Importante saber interpretar esta métrica, ter um Distinct-1 baixo não significa ter um dataset fraco. Depende do objetivo e contexto. <p>

In [41]:
palavras = []

for i in df_phi["query"]:
    palavras.extend (i.split())

    distinct_1 = len(set(palavras)) / len(palavras)

print (f"Distinct-1 Dataset Phi -> {distinct_1:.2f}")

## ----------------------------------------------------------- ##

palavras = []

for i in df_mistral["query"]:
    palavras.extend (i.split())

    distinct_1 = len(set(palavras)) / len(palavras)

print (f"Distinct-1 Dataset Mistral -> {distinct_1:.2f}")

Distinct-1 Dataset Phi -> 0.35
Distinct-1 Dataset Mistral -> 0.28


<hr>

<h1> Similaridade Semântica </h1>

<p> Mede a similaridade entre perguntas, quanto menor neste caso, melhor! </p>

In [42]:
BI_ENCODER = r"C:\Users\Admin\Desktop\models\Bi Encoder" #sentence-transformers/all-MiniLM-L6-v2

TOKENIZER = AutoTokenizer.from_pretrained (BI_ENCODER)
MODEL = AutoModel.from_pretrained (BI_ENCODER, device_map = device)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1008.31it/s]


In [43]:
querys = []

for i in df_phi["query"]:
    querys.append(i) 


inputs = TOKENIZER (querys, return_tensors = "pt", padding = True, truncation = True).to(device)

with torch.no_grad():
    outputs = MODEL (**inputs)

#Pooling é o processo de juntar vários vetores (neste caso por token) num único vetor fixo que representa todo o texto.
#Isto é importante porque neste momento temos apenas vetores por cada token do respetivo chunk e nós queremos uma representação final do chunk.
mask = inputs ["attention_mask"].unsqueeze (-1) #Não podemos ter em conta os tokens [PAD]
embedding = (outputs.last_hidden_state * mask).sum(dim = 1) / mask.sum(dim = 1)

embedding = F.normalize (embedding, p = 2, dim = 1) #Normalização! Importante

#print (embedding.shape)
#print (embedding)
#print (len(embedding))

# Similaridade do Cosseno
cos_sim = cosine_similarity (embedding.cpu())

print (f"{cos_sim.mean():.2f}")
#print (cos_sim.shape)
#print (cos_sim)
#print (len(cos_sim))

0.47


In [44]:
querys = []

for i in df_mistral["query"]:
    querys.append(i) 


inputs = TOKENIZER (querys, return_tensors = "pt", padding = True, truncation = True).to(device)

with torch.no_grad():
    outputs = MODEL (**inputs)

#Pooling é o processo de juntar vários vetores (neste caso por token) num único vetor fixo que representa todo o texto.
#Isto é importante porque neste momento temos apenas vetores por cada token do respetivo chunk e nós queremos uma representação final do chunk.
mask = inputs ["attention_mask"].unsqueeze (-1) #Não podemos ter em conta os tokens [PAD]
embedding = (outputs.last_hidden_state * mask).sum(dim = 1) / mask.sum(dim = 1)

embedding = F.normalize (embedding, p = 2, dim = 1) #Normalização! Importante

#print (embedding.shape)
#print (embedding)
#print (len(embedding))

# Similaridade do Cosseno
cos_sim = cosine_similarity (embedding.cpu())

print (f"{cos_sim.mean():.2f}")
#print (cos_sim.shape)
#print (cos_sim)
#print (len(cos_sim))

0.44


<hr>

Podem ser adicionadas mais métricas